# Week 2 Assignment: End-to-End Data Wrangling & Exploratory Data Analysis (EDA)

In [1]:
import pandas as pd
import numpy as np

# Task 1: Data Ingestion & Structural Audit

In [5]:
telemetry = pd.read_csv("/Users/shreeyabogati/week 2/Assignment 2/saas_telemetry_raw.csv")
telemetry.head()

,user_id,signup_date,region,subscription_tier,monthly_usage_hours,monthly_bill
0,UID_10000,2024-01-01,EMEA,Pro,32.3,33.9
1,UID_10001,2024-01-01,APAC,Basic,28.2,10.32
2,UID_10002,2024-01-02,APAC,Enterprise,284.9,311.52
3,UID_10003,2024-01-02,North America,Basic,32.4,14.87
4,UID_10004,2024-01-03,EMEA,Pro,68.5,30.99


In [6]:
print("Shape:", telemetry.shape)
telemetry.dtypes

Shape: (300, 6)


user_id                str
signup_date            str
region                 str
subscription_tier      str
monthly_usage_hours    str
monthly_bill           str
dtype: object

In [7]:
telemetry.isna().sum()

user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    2
monthly_bill           1
dtype: int64

In [8]:
disguised_nulls = ["n/a", "missing", "-999"]

for col in telemetry.columns:
    values_seen = telemetry[col].dropna().astype(str).str.lower().unique()
    hits = [flag for flag in disguised_nulls if flag in values_seen]
    if hits:
        print(f"{col}: hides {hits}")


monthly_usage_hours: hides ['missing', '-999']
monthly_bill: hides ['missing']


# Task 2: Data Cleaning & Quality Enforcement
Standardizing the hidden null flags

In [13]:
elemetry = pd.read_csv(
    "/Users/shreeyabogati/week 2/Assignment 2/saas_telemetry_raw.csv",
    na_values=["N/A", "missing", "-999"]
)

telemetry["monthly_usage_hours"] = pd.to_numeric(telemetry["monthly_usage_hours"], errors="coerce")
telemetry["monthly_bill"] = pd.to_numeric(telemetry["monthly_bill"], errors="coerce")

telemetry.isna().sum()


user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    3
monthly_bill           2
dtype: int64

Fixing the signup_date type

In [14]:
telemetry["signup_date"] = pd.to_datetime(telemetry["signup_date"])
telemetry["signup_date"].head()

0   2024-01-01
1   2024-01-01
2   2024-01-02
3   2024-01-02
4   2024-01-03
Name: signup_date, dtype: datetime64[us]

Correcting the negative usage bug

In [16]:
bad_usage_rows = telemetry["monthly_usage_hours"] < 0
telemetry.loc[bad_usage_rows, ["user_id", "monthly_usage_hours"]]

,user_id,monthly_usage_hours
8,UID_10008,-15.5
112,UID_10112,-999.0
189,UID_10189,-42.0


In [17]:
telemetry.loc[bad_usage_rows, "monthly_usage_hours"] = np.nan
(telemetry["monthly_usage_hours"] < 0).sum()

np.int64(0)

Filling the remaining gaps by subscription tier

In [18]:
tier_typical_usage = telemetry.groupby("subscription_tier")["monthly_usage_hours"].transform("median")

telemetry["monthly_usage_hours"] = telemetry["monthly_usage_hours"].fillna(tier_typical_usage)
telemetry["monthly_usage_hours"].isna().sum()


np.int64(0)

# Task 3: Feature Engineering with Group Transformations

In [19]:
telemetry["tier_avg_usage"] = telemetry.groupby("subscription_tier")["monthly_usage_hours"].transform("mean")
telemetry[["subscription_tier", "monthly_usage_hours", "tier_avg_usage"]].head()


,subscription_tier,monthly_usage_hours,tier_avg_usage
0,Pro,32.3,73.736111
1,Basic,28.2,22.910959
2,Enterprise,284.9,232.378261
3,Basic,32.4,22.910959
4,Pro,68.5,73.736111


In [20]:
telemetry["usage_deviation"] = telemetry["monthly_usage_hours"] - telemetry["tier_avg_usage"]
telemetry[["user_id", "subscription_tier", "usage_deviation"]].sort_values("usage_deviation", ascending=False).head()

,user_id,subscription_tier,usage_deviation
253,UID_10253,Enterprise,110.221739
111,UID_10111,Enterprise,108.321739
40,UID_10040,Enterprise,105.821739
25,UID_10025,Enterprise,104.121739
28,UID_10028,Enterprise,96.421739


# Task 4: Executive Aggregations (Split-Apply-Combine)

In [21]:
executive_summary = telemetry.groupby(["subscription_tier", "region"]).agg(
    total_users=("user_id", "nunique"),
    avg_monthly_bill=("monthly_bill", "mean"),
    total_usage_hours=("monthly_usage_hours", "sum"),
    max_usage_deviation=("usage_deviation", "max")
).reset_index()   # turns the tier/region group labels back into normal columns

executive_summary


,subscription_tier,region,total_users,avg_monthly_bill,total_usage_hours,max_usage_deviation
0,Basic,APAC,32,12.347188,706.05,16.289041
1,Basic,EMEA,37,12.676486,836.70,16.989041
2,Basic,LATAM,44,12.571818,989.90,16.189041
3,Basic,North America,33,12.915455,812.35,16.489041
4,Enterprise,APAC,11,390.780000,2509.30,87.121739
5,Enterprise,EMEA,12,315.526667,3104.30,110.221739
6,Enterprise,LATAM,11,345.003000,2611.80,84.621739
7,Enterprise,North America,12,405.699091,2464.00,105.821739
8,Pro,APAC,19,39.588421,1327.70,35.463889
9,Pro,EMEA,28,38.621071,1840.10,25.363889


# Task 5: Reshaping for Business Reporting

In [22]:
billing_by_tier_and_region = pd.pivot_table(
    telemetry,
    values="monthly_bill",
    index="subscription_tier",
    columns="region",
    aggfunc="mean",
    margins=True   # adds an overall "All" row and column
)

billing_by_tier_and_region.round(2)

region,APAC,EMEA,LATAM,North America,All
subscription_tier,,,,,
Basic,12.35,12.68,12.57,12.92,12.63
Enterprise,390.78,315.53,345.00,405.70,363.58
Pro,39.59,38.62,41.22,40.77,40.10
All,87.84,69.31,64.61,78.06,74.40
